Let's start with yesterday's handtracker as a basis

In [4]:

import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
import cv2 as cv


def main(frame_width=640, frame_height=480):

    
    # Initialize the camera 
    camera = cv2.VideoCapture(0)

    # Initialize the hand landmarker
    baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
    options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=2)
    detector = vision.HandLandmarker.create_from_options(options)

    # Check if the camera opened successfully
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        # Start the main loop to read frames from the camera
        while True:

            # Read a frame from the camera
            ret, frame = camera.read()

            # Check if the frame was read successfully
            if not ret:
                print("Error: Could not read frame.")
                break

            # Resize the frame and convert it to RGB for MediaPipe processing
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            # Create a MediaPipe Image object from the RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Detect hands in the frame using the hand landmarker
            result = detector.detect(mp_image)

            # If hand landmarks are detected, process them to calculate the average position and draw on the frame
            if result.hand_landmarks:
                # Iterate through each detected hand's landmarks
                for hand_landmarks in result.hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)

            
            # Display the processed frame in a window titled 'Hand Detection'
            cv2.imshow('Hand Detection', frame)\
            
            # Press "Q" to exit the loop and close the application
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


Now let's extract the position of the tips of each finger. This way, we can use that date to determine which fingers are extended.

In [5]:

import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
import cv2 as cv


def main(frame_width=640, frame_height=480, image_path="image.jpg"):

    
    # Initialize the camera 
    camera = cv2.VideoCapture(0)

    # Initialize the hand landmarker
    baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
    options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=2)
    detector = vision.HandLandmarker.create_from_options(options)

    # Check if the camera opened successfully
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        # Start the main loop to read frames from the camera
        while True:

            # Read a frame from the camera
            ret, frame = camera.read()

            # Check if the frame was read successfully
            if not ret:
                print("Error: Could not read frame.")
                break

            # Resize the frame and convert it to RGB for MediaPipe processing
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            # Create a MediaPipe Image object from the RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Detect hands in the frame using the hand landmarker
            result = detector.detect(mp_image)

            # If hand landmarks are detected, process them to calculate the average position and draw on the frame
            if result.hand_landmarks:
                # Iterate through each detected hand's landmarks
                for hand_landmarks in result.hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                    # NEW: iterate through each landmark and put the number of the landmark next to it
                    for i in range(0, len(hand_landmarks)):
                        landmark = hand_landmarks[i]
                        cv2.putText(
                            frame, 
                            f"{i}", 
                            (int(landmark.x*frame_width), int(landmark.y*frame_height)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 
                            0.7, 
                            (255,255,0), 
                            2)

            
            # Display the processed frame in a window titled 'Hand Detection'
            cv2.imshow('Hand Detection', frame)\
            
            # Press "Q" to exit the loop and close the application
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


Now we modify it so that we can create our own dataset this way. I have seen other projects that use pre-existing datasets. However, I wanted to try a different approach. As such, in this approach, you can generate a dataset by making a gesture, and pressing a number to associate with it. This will save the landmarks to a CSV file, which then can be used to train a neural network.

Hypothetically, this is more efficient than isolating each hand and running image recognition on them; rather than using a large image as input, only 20 tuples will be used per hand, allowing for a much smaller neural network.

In [6]:

import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
import cv2 as cv
import pickle

def build_training_dataset(frame_width=640, frame_height=480, image_path="image.jpg"):
    gestures = {
        1:[],
        2:[],
        3:[],
        4:[],
        5:[],
    }
    
    # Initialize the camera 
    camera = cv2.VideoCapture(0)

    # Initialize the hand landmarker
    baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
    options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=1) #NEW: Switch to one hand for simplicity.
    detector = vision.HandLandmarker.create_from_options(options)

    # Check if the camera opened successfully
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        # Start the main loop to read frames from the camera
        while True:
            landmark_pos = {}
            # Read a frame from the camera
            ret, frame = camera.read()

            # Check if the frame was read successfully
            if not ret:
                print("Error: Could not read frame.")
                break

            # Resize the frame and convert it to RGB for MediaPipe processing
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            # Create a MediaPipe Image object from the RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Detect hands in the frame using the hand landmarker
            result = detector.detect(mp_image)

            # If hand landmarks are detected, process them to calculate the average position and draw on the frame
            if result.hand_landmarks:
                # Iterate through each detected hand's landmarks
                for hand_landmarks in result.hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                    # Iterate through each landmark and put the number of the landmark next to it
                    for i in range(0, len(hand_landmarks)):
                        landmark = hand_landmarks[i]
                        cv2.putText(
                            frame, 
                            f"{i}", 
                            (int(landmark.x*frame_width), int(landmark.y*frame_height)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 
                            0.7, 
                            (255,255,0), 
                            2)
                        landmark_pos[i] = {
                            "x": landmark.x,
                            "y": landmark.y,
                            "z": landmark.z,
                        }

            
            # Display the processed frame in a window titled 'Hand Detection'
            cv2.imshow('Hand Detection', frame)\
            
            # NEW: capture key input and save to the key variable
            key = cv2.waitKey(1)

            # Press "Q" to exit the loop and close the application
            if key == ord("q"):
                print(gestures)
                with open("gestures.pickle", "wb") as file:
                    pickle.dump(gestures, file)
                break
            elif key == ord("1"):
                gestures[1].append(landmark_pos)
            elif key == ord("2"):
                gestures[2].append(landmark_pos)
            elif key == ord("3"):
                gestures[3].append(landmark_pos)
            elif key == ord("4"):
                gestures[4].append(landmark_pos)
            elif key == ord("5"):
                gestures[5].append(landmark_pos)
            else:
                pass     
    except KeyboardInterrupt:
        pass

# build_training_dataset()

Now we have the dataset. So let's train a model on it. A fully connected neural network was chosen, to efficiently deal with the 63 (21 landmarks* 3 values for the coordinates) inputs and translate them to 5 outputs.

So first, we make the model

In [7]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(63, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 5)
        )

    def forward(self, x):
        return self.net(x)
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(63, 120)
        self.fc2 = nn.Linear(120, 30)
        self.fc3 = nn.Linear(30, 5)

        self.activator = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        # (batch, 21, 3) -> (batch, 63)
        x = torch.flatten(x, start_dim=1)

        x = self.dropout(self.activator(self.fc1(x)))
        x = self.dropout(self.activator(self.fc2(x)))
        x = self.fc3(x)

        return x

Next, we make a dataset.

In [8]:
from torch.utils.data import Dataset
import pickle 

class HandLandmarksDataset(Dataset):
    def __init__(self, pickle_file):
        with open(pickle_file, "rb") as f:
            data = pickle.load(f)

        self.samples = []

        self.class_names = sorted(data.keys())
        self.class_to_idx = {
            name: i
            for i, name in enumerate(self.class_names)
        }

        skipped = 0

        for class_name, constellation_list in data.items():
            label = self.class_to_idx[class_name]

            for constellation in constellation_list:
                if len(constellation) != 21:
                    skipped += 1
                    continue

                self.samples.append((constellation, label))

        print(f"Loaded {len(self.samples)} samples.")
        print(f"Skipped {skipped} invalid samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        constellation, label = self.samples[idx]

        if len(constellation) != 21:
            raise ValueError(
                f"Sample {idx} has {len(constellation)} landmarks instead of 21."
            )

        points = [
            [constellation[i]["x"], constellation[i]["y"], constellation[i]["z"]]
            for i in range(21)
        ]
        points = torch.tensor(points, dtype=torch.float32)

        points = (points - points.mean(dim=0)) / (points.std(dim=0) + 1e-6)

        x = torch.tensor(points, dtype=torch.float32)
        y = torch.tensor(label, dtype=torch.long)

        return x, y
    
dataset = HandLandmarksDataset("gestures.pickle")

Loaded 218 samples.
Skipped 33 invalid samples.


Now create the training, testing, and validation dataset

In [9]:
import torch
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_set, val_set, test_set = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False
)

Train the model

In [10]:
import torch.optim as optim

def one_epoch(model, train_data_loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_data_loader:
        inputs, labels = inputs.to(device), labels.to(device)
 
        # Zero the parameter gradients
        optimizer.zero_grad()
 
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
 
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
 
    epoch_loss = running_loss / len(train_data_loader)
    return epoch_loss

def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            # Forward pass
            outputs = model(x)
            # Compute loss
            loss = criterion(outputs, y)
            running_loss += loss.item() * x.size(0)

            # Compute predictions
            _, predicted = torch.max(outputs, dim=1)

            correct += (predicted == y).sum().item()
            total += y.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

def train(epochs, model, train_loader, val_loader, test_loader):
    criterion = nn.CrossEntropyLoss()
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.to(device)
    best_model = None
    best_val_loss = 9999
    for epoch in range(0, epochs):
        train_loss = one_epoch(model, train_data_loader=train_loader, optimizer=optimizer, criterion=criterion, device=device)
        val_loss, val_accuracy = evaluate(model, val_loader, criterion=criterion, device=device)
        if val_loss < best_val_loss:
            best_model = model
        print(f"Epoch: {epoch} | Train Loss: {train_loss} | Val Loss: { val_loss} | Accuracy: {val_accuracy}")
    test_loss, test_accuracy = evaluate(model, test_loader, criterion=criterion, device=device)
    print(f"Test | Loss: {test_loss} | Accuracy: {test_accuracy}")
    return best_model

# model = Model()
# model = train(60, model=model, train_loader=train_loader, test_loader=test_loader, val_loader=val_loader)

Now that we have our model, we create a new main function that uses this model to predict the user's gesture

In [11]:

import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
import cv2 as cv


def main(model, frame_width=640, frame_height=480):

    
    # Initialize the camera 
    camera = cv2.VideoCapture(0)

    # Initialize the hand landmarker
    baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
    options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=2)
    detector = vision.HandLandmarker.create_from_options(options)

    # Check if the camera opened successfully
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        # Start the main loop to read frames from the camera
        while True:
            landmark_pos = {}
            # Read a frame from the camera
            ret, frame = camera.read()

            # Check if the frame was read successfully
            if not ret:
                print("Error: Could not read frame.")
                break

            # Resize the frame and convert it to RGB for MediaPipe processing
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            # Create a MediaPipe Image object from the RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Detect hands in the frame using the hand landmarker
            result = detector.detect(mp_image)

            # If hand landmarks are detected, process them to calculate the average position and draw on the frame
            if result.hand_landmarks:
                # Iterate through each detected hand's landmarks
                for hand_landmarks in result.hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                    # NEW: iterate through each landmark and put the number of the landmark next to it
                    for i in range(0, len(hand_landmarks)):
                        landmark = hand_landmarks[i]
                        cv2.putText(
                            frame, 
                            f"{i}", 
                            (int(landmark.x*frame_width), int(landmark.y*frame_height)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 
                            0.7, 
                            (255,255,0), 
                            2) 
                        landmark_pos[i] = {
                                        "x": landmark.x,
                                        "y": landmark.y,
                                        "z": landmark.z,
                                    }

            if len(landmark_pos) == 21:
                points = [
                    [landmark_pos[i]["x"], landmark_pos[i]["y"], landmark_pos[i]["z"]]
                    for i in range(21)
                ]

                x = torch.tensor(points, dtype=torch.float32).unsqueeze(0)

                with torch.no_grad():
                    prediction = model(x)
                    prediction = torch.argmax(prediction, dim=1).item()
            else:
                prediction = None
            print(prediction)
            cv2.putText(frame, str(prediction), (0,0), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
            # Display the processed frame in a window titled 'Hand Detection'
            cv2.imshow('Hand Detection', frame)\
            
            # Press "Q" to exit the loop and close the application
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


This approach consistently misclassifies the gestures. So let's try image recognition instead.

First, we bring back our old main function.

In [12]:

import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
import cv2 as cv


def main(frame_width=640, frame_height=480, image_path="image.jpg"):

    
    # Initialize the camera 
    camera = cv2.VideoCapture(0)

    # Initialize the hand landmarker
    baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
    options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=2)
    detector = vision.HandLandmarker.create_from_options(options)

    # Check if the camera opened successfully
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        # Start the main loop to read frames from the camera
        while True:

            # Read a frame from the camera
            ret, frame = camera.read()

            # Check if the frame was read successfully
            if not ret:
                print("Error: Could not read frame.")
                break

            # Resize the frame and convert it to RGB for MediaPipe processing
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            # Create a MediaPipe Image object from the RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Detect hands in the frame using the hand landmarker
            result = detector.detect(mp_image)

            # If hand landmarks are detected, process them to calculate the average position and draw on the frame
            if result.hand_landmarks:
                # Iterate through each detected hand's landmarks
                for hand_landmarks in result.hand_landmarks:
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                    # NEW: iterate through each landmark and put the number of the landmark next to it
                    for i in range(0, len(hand_landmarks)):
                        landmark = hand_landmarks[i]
                        cv2.putText(
                            frame, 
                            f"{i}", 
                            (int(landmark.x*frame_width), int(landmark.y*frame_height)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 
                            0.7, 
                            (255,255,0), 
                            2)

            
            # Display the processed frame in a window titled 'Hand Detection'
            cv2.imshow('Hand Detection', frame)\
            
            # Press "Q" to exit the loop and close the application
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    except KeyboardInterrupt:
        pass


Now let's create a bounding box around the hand, and stream this to the notebook.

In [13]:

import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
import cv2 as cv
from IPython.display import clear_output, Image, display, HTML

def main(frame_width=640, frame_height=480, image_path="image.jpg"):

    
    # Initialize the camera 
    camera = cv2.VideoCapture(0)

    # Initialize the hand landmarker
    baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
    options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=2)
    detector = vision.HandLandmarker.create_from_options(options)

    # Check if the camera opened successfully
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        # Start the main loop to read frames from the camera
        while True:

            # Read a frame from the camera
            ret, frame = camera.read()

            # Check if the frame was read successfully
            if not ret:
                print("Error: Could not read frame.")
                break

            # Resize the frame and convert it to RGB for MediaPipe processing
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            # Create a MediaPipe Image object from the RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Detect hands in the frame using the hand landmarker
            result = detector.detect(mp_image)

            # If hand landmarks are detected, process them to calculate the average position and draw on the frame
            if result.hand_landmarks:
                # Iterate through each detected hand's landmarks
                for hand_landmarks in result.hand_landmarks:

                    
                    
                    xvals = [int(landmark.x * frame_width) for landmark in hand_landmarks]
                    yvals = [int(landmark.y * frame_height) for landmark in hand_landmarks]
                    xmax = max(xvals)
                    ymax = max(yvals)
                    xmin = min(xvals)
                    ymin = min(yvals)

                    padding = int(max(xmax-xmin, ymax-ymin)*0.2)

                    xmax = min(frame_width, xmax+padding)
                    ymax = min(frame_height, ymax+padding)
                    xmin = max(0, xmin-padding)
                    ymin = max(0, ymin-padding)
                    cv2.rectangle(
                        frame,
                        (xmin, ymin),
                        (xmax, ymax),
                        (0,255,0), 
                        2
                    )
                    hand_crop = frame[ymin:ymax, xmin:xmax]
                    _, buffer = cv2.imencode('.jpg', hand_crop)
                    img_data = buffer.tobytes()
                    clear_output(wait=True)
                    display(Image(data=img_data))
                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                    for i in range(0, len(hand_landmarks)):
                        landmark = hand_landmarks[i]
                        cv2.putText(
                            frame, 
                            f"{i}", 
                            (int(landmark.x*frame_width), int(landmark.y*frame_height)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 
                            0.7, 
                            (255,255,0), 
                            2)
            # Display the processed frame in a window titled 'Hand Detection'
            cv2.imshow('Hand Detection', frame)\
            
            # Press "Q" to exit the loop and close the application
            if cv2.waitKey(1) & 0xFF == ord('q'):
                camera.release()
                cv2.destroyAllWindows
                break
    except KeyboardInterrupt:
        
        pass


Now we need a classifier model. For that I am using MobileNetV2. MobileNetV2 is a lightweight model, capable on running on edge devices such as phones, cameras, and even Arduinos. Despite that, they performed almost as well as the more popular ResNet8 model in my previous research at Hendrix. Taking these two facts into account, we make the following function.

We use the dataset https://www.kaggle.com/datasets/barnabaspeter/hand-gesture-dataset for this project, and as such we use 7 classes.

In [34]:
from torchvision import models
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score


def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            running_loss += criterion(outputs, labels).item()

            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    loss = running_loss / len(loader)
    accuracy = (np.array(all_preds) == np.array(all_labels)).mean()
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return loss, accuracy, macro_f1

def train_model(num_classes = 7, path = r"C:\\Users\\dstra\\portfolio\\gesture_recognition\\dataset", epochs = 15):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(
        model.last_channel,
        num_classes
    )
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(15),
        transforms.ToTensor(),
    ])
    dataset = ImageFolder(path, transform=train_transform)
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size

    train_set, val_set, test_set = random_split(
        dataset,
        [train_size, val_size, test_size]
    )

    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32)
    test_loader = DataLoader(test_set, batch_size=32)

    classes = train_loader.dataset.dataset.classes

    optimizer = optim.AdamW(
                model.parameters(), 
                lr=0.001, 
                weight_decay=0.02
    
            )
    criterion = nn.CrossEntropyLoss()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)
    best_loss = 9999
    best_model = None
    for epoch in range(0,epochs-1):
        model.train()
        running_loss = 0.0

        for i, data in enumerate(train_loader, 0):
            inputs, labels = data[0].to(device), data[1].to(device)
    
            optimizer.zero_grad()
    
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)

        val_loss, val_acc, val_f1 = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        if val_loss < best_loss:
            best_loss = best_loss
            best_model = model

        print(
            f"Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val F1: {val_f1:.4f}"
        )

    test_loss, test_acc, test_f1 = evaluate(
        best_model,
        test_loader,
        criterion,
        device
    )

    print(
        f"Test | "
        f"Loss: {test_loss:.4f} | "
        f"Accuracy: {test_acc:.4f} | "
        f"Macro F1: {test_f1:.4f}"
    )


    return best_model, classes

model, classes = train_model(6)

Epoch 1 | Train Loss: 0.5172 | Val Loss: 0.3750 | Val Acc: 0.8958 | Val F1: 0.7895
Epoch 2 | Train Loss: 0.0049 | Val Loss: 0.0354 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 3 | Train Loss: 0.0015 | Val Loss: 0.0163 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 4 | Train Loss: 0.0009 | Val Loss: 0.0060 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 5 | Train Loss: 0.0005 | Val Loss: 0.0003 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 6 | Train Loss: 0.0003 | Val Loss: 0.0001 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 7 | Train Loss: 0.0003 | Val Loss: 0.0001 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 8 | Train Loss: 0.0004 | Val Loss: 0.0001 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 9 | Train Loss: 0.0002 | Val Loss: 0.0000 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 10 | Train Loss: 0.0003 | Val Loss: 0.0000 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 11 | Train Loss: 0.0003 | Val Loss: 0.0000 | Val Acc: 1.0000 | Val F1: 1.0000
Epoch 12 | Train Loss: 0.0002 | Val Loss: 0.0000 | Val Acc: 1.0000 | Val F1: 1.0000
E

Save the model and the classes to avoid having to retrain the model each run

In [39]:
import pickle

with open("model.pickle", "wb") as file:
    pickle.dump(model, file)

print(classes)

['01_palm', '02_fist', '03_thumbs-up', '04_thumbs-down', '05_index-right', '06_index-left', '07_no-gesture']


In [40]:
with open("model.pickle", "rb") as file:
    model = pickle.load(file)

classes = ['01_palm', '02_fist', '03_thumbs-up', '04_thumbs-down', '05_index-right', '06_index-left', '07_no-gesture']

Now we run classification on the hand and estimate the associated gesture

In [41]:

import cv2
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils as mp_drawing
import mediapipe as mp
import cv2 as cv
from IPython.display import clear_output, Image, display, HTML

def main(model,classes, frame_width=640, frame_height=480):

    
    # Initialize the camera 
    camera = cv2.VideoCapture(0)

    # Initialize the hand landmarker
    baseOptions = python.BaseOptions(model_asset_path="hand_landmarker.task")
    options = vision.HandLandmarkerOptions(base_options=baseOptions, num_hands=2)
    detector = vision.HandLandmarker.create_from_options(options)


    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    
    # Check if the camera opened successfully
    if not camera.isOpened():
        print("Error: Could not open camera.")
    try:
        # Start the main loop to read frames from the camera
        while True:

            # Read a frame from the camera
            ret, frame = camera.read()

            # Check if the frame was read successfully
            if not ret:
                print("Error: Could not read frame.")
                break

            # Resize the frame and convert it to RGB for MediaPipe processing
            frame = cv2.resize(frame, (frame_width, frame_height))
            rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

            # Create a MediaPipe Image object from the RGB frame
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            
            # Detect hands in the frame using the hand landmarker
            result = detector.detect(mp_image)

            # If hand landmarks are detected, process them to calculate the average position and draw on the frame
            if result.hand_landmarks:
                # Iterate through each detected hand's landmarks
                for hand_landmarks in result.hand_landmarks:

                    
                    
                    xvals = [int(landmark.x * frame_width) for landmark in hand_landmarks]
                    yvals = [int(landmark.y * frame_height) for landmark in hand_landmarks]
                    xmax = max(xvals)
                    ymax = max(yvals)
                    xmin = min(xvals)
                    ymin = min(yvals)

                    padding = int(max(xmax-xmin, ymax-ymin)*0.2)

                    xmax = min(frame_width, xmax+padding)
                    ymax = min(frame_height, ymax+padding)
                    xmin = max(0, xmin-padding)
                    ymin = max(0, ymin-padding)
                    cv2.rectangle(
                        frame,
                        (xmin, ymin),
                        (xmax, ymax),
                        (0,255,0), 
                        2
                    )
                    hand_crop = frame[ymin:ymax, xmin:xmax]
                    hand_crop = cv2.cvtColor(hand_crop, cv2.COLOR_BGR2RGB)
                    x = transform(hand_crop)
                    x = x.unsqueeze(0)
                    with torch.no_grad():
                        logits = model(x)
                    certainties = torch.softmax(logits, dim=1)
                    pred_idx = torch.argmax(certainties, dim=1).item()
                    confidence = certainties[0, pred_idx].item()
                    pred_class = classes[pred_idx]

                    cv2.putText(frame, 
                                f"{pred_class} : {(confidence * 100):.2f}%", 
                                (xmax, ymax),
                                cv2.FONT_HERSHEY_SIMPLEX, 
                                0.7, 
                                (255,255,0), 
                                2
                            )

                    mp_drawing.draw_landmarks(frame, hand_landmarks, vision.HandLandmarksConnections.HAND_CONNECTIONS)
                    for i in range(0, len(hand_landmarks)):
                        landmark = hand_landmarks[i]
                        cv2.putText(
                            frame, 
                            f"{i}", 
                            (int(landmark.x*frame_width), int(landmark.y*frame_height)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 
                            0.2, 
                            (255,255,0), 
                            2)
            # Display the processed frame in a window titled 'Hand Detection'
            cv2.imshow('Hand Detection', frame)\
            
            # Press "Q" to exit the loop and close the application
            if cv2.waitKey(1) & 0xFF == ord('q'):
                camera.release()
                cv2.destroyAllWindows
                break
    except KeyboardInterrupt:
        
        pass


This dataset consistently gives false returns. Different dataset?

In [42]:
main(model,classes)